# Prior Authorization Readiness Classifier

Binary text classification: given a de-identified lab event with its diagnosis
and requested procedure, predict whether a prior authorization (PA) request is
**Ready** or **Not Ready** to submit.

The notebook builds leakage-free notes from structured MIMIC-IV fields, fits a
TF-IDF + logistic regression baseline, fine-tunes three biomedical BERT models
with the same settings, then compares them with ROC/PR curves, a metric table
and error analysis. Results and what they mean are in the README.

## Setup
Restart the runtime after installing so the new packages load.

In [ ]:
# Install dependencies
# Run this cell first, then: Runtime → Restart runtime
!pip -q install -U transformers datasets accelerate evaluate scikit-learn openpyxl matplotlib seaborn

## Data
5,000 de-identified lab event records from MIMIC-IV with 24 columns. The key
fields are `Clinical_Comments`, `PriorAuth_Status`, `valuenum`,
`ref_range_lower`, `ref_range_upper`, `flag` and `priority`. The dataset is not
included in this repo because MIMIC-IV access requires a PhysioNet data use
agreement.

In [ ]:
# Load data
# Place the dataset at data/ClinicalDataProject.xlsx, or upload it when running in Colab.
import os
import pandas as pd

DATA_PATH = "data/ClinicalDataProject.xlsx"
if not os.path.exists(DATA_PATH):
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = next(iter(uploaded.keys()))

df = pd.read_excel(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

## Feature engineering
`Clinical_Comments` is built from template sentences such as "Requesting prior
authorization for ..." that track the label closely. Feeding it in directly
would let a model memorize the template instead of the clinical content, so
only the diagnosis and procedure names are pulled out with regex.

Each row is then rewritten as a short structured note:

```
Diagnosis Hypertension. Procedure Echocardiogram. Lab item 50868 value 1.2 mg/dL.
Reference range 0.6 to 1.3. Severity within normal range. Flag abnormal. Priority routine.
```

Severity encodes how far a value is outside its reference range (more than 30%
of the range width counts as critical). Flag and priority are lowercased so each
concept maps to one token. The last step checks that none of the template
phrases survive in the notes.

In [ ]:
# Feature Engineering (severity encoding + flag normalization)
import re
import numpy as np

TEXT_COL  = "Clinical_Comments"
LABEL_COL = "PriorAuth_Status"

# ── Label encoding ─────────────────────────────────────────────────────────────
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()
label_map = {"Not Ready": 0, "Ready": 1}
df["label"] = df[LABEL_COL].map(label_map)
df = df.dropna(subset=["label"]).copy()
df["label"] = df["label"].astype(int)

# ── Regex extractors (leakage-free) ───────────────────────────────────────────
def extract_diag(text):
    text = "" if text is None else str(text)
    m = re.search(r"(?i)patient (?:diagnosed with|with history of)\s+([^\.]+)\.", text)
    return m.group(1).strip() if m else ""

def extract_proc(text):
    text = "" if text is None else str(text)
    m = re.search(r"(?i)requesting prior authorization for\s+([^\.]+)\.", text)
    if m: return m.group(1).strip()
    m = re.search(r"(?i)considering\s+([^\.]+?)(?:\s+for management|\s+screening)?\.", text)
    return m.group(1).strip() if m else ""

df["diag"] = df[TEXT_COL].apply(extract_diag)
df["proc"] = df[TEXT_COL].apply(extract_proc)

# ── Range status ───────────────────────────────────────────────────────────────
def range_status(row):
    try:
        v  = float(row["valuenum"])
        lo = float(row["ref_range_lower"])
        hi = float(row["ref_range_upper"])
        if np.isnan(v) or np.isnan(lo) or np.isnan(hi): return "unknown"
        if v < lo: return "below"
        if v > hi: return "above"
        return "within"
    except: return "unknown"

df["range_status"] = df.apply(range_status, axis=1)

# ── Severity encoding ──────────────────────────────────────────────────────────
# >30% outside range = "critically"; any deviation = "mildly"; within = "within normal"
def severity_phrase(row):
    try:
        v  = float(row["valuenum"])
        lo = float(row["ref_range_lower"])
        hi = float(row["ref_range_upper"])
        if np.isnan(v) or np.isnan(lo) or np.isnan(hi): return ""
        range_width = hi - lo if (hi - lo) > 0 else 1
        if v < lo:
            pct_off = (lo - v) / range_width
            return "critically below range" if pct_off > 0.3 else "mildly below range"
        elif v > hi:
            pct_off = (v - hi) / range_width
            return "critically above range" if pct_off > 0.3 else "mildly above range"
        return "within normal range"
    except: return ""

df["severity"] = df.apply(severity_phrase, axis=1)

# ── Flag & priority normalization ──────────────────────────────────────────────
df["flag_clean"]     = df["flag"].fillna("normal").astype(str).str.lower().str.strip()
df["priority_clean"] = df["priority"].fillna("routine").astype(str).str.lower().str.strip()

# ── Build structured note_text ─────────────────────────────────────────────────
def build_note(row):
    diag     = (row["diag"] or "unknown diagnosis").strip()
    proc     = (row["proc"] or "unknown procedure").strip()
    itemid   = row.get("itemid", "")
    val      = row.get("valuenum", "")
    uom      = row.get("valueuom", "")
    lo       = row.get("ref_range_lower", "")
    hi       = row.get("ref_range_upper", "")
    flag     = row.get("flag_clean", "normal")
    prio     = row.get("priority_clean", "routine")
    rs       = row.get("range_status", "unknown")
    severity = row.get("severity", "")
    sev_str  = f"Severity {severity}." if severity else f"Range status {rs}."
    return (
        f"Diagnosis {diag}. Procedure {proc}. "
        f"Lab item {itemid} value {val} {uom}. Reference range {lo} to {hi}. "
        f"{sev_str} Flag {flag}. Priority {prio}."
    )

df["note_text"] = df.apply(build_note, axis=1)
df = df[df["note_text"].str.len() > 50].copy()

print("Rows after prep:", df.shape)
print("Label distribution:\n", df["label"].value_counts())

# ── Leakage sanity check ───────────────────────────────────────────────────────
leak_phrases = [
    "documentation incomplete", "medical necessity criteria",
    "requesting prior authorization", "objective findings include",
    "symptoms persist despite",
]
t = df["note_text"].str.lower()
print("\nLeak phrase rates in note_text (should be ~0):")
for p in leak_phrases:
    print(f"  {p} -> {float(t.str.contains(p, na=False).mean()):.4f}")

## Train / validation / test split
Duplicates are dropped first, then a stratified 70/15/15 split (3,430 / 735 / 736).
The validation set is used for checkpoint selection and threshold tuning only.

In [ ]:
# Train / Val / Test Split
from sklearn.model_selection import train_test_split

df = df.drop_duplicates(subset=['note_text']).copy()

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['label']
)

print(f'Train: {train_df.shape}  Val: {val_df.shape}  Test: {test_df.shape}')
print('Test label distribution:\n', test_df['label'].value_counts())

## Baseline: TF-IDF + logistic regression
Unigrams and bigrams, 15,000 features, balanced class weights. Every transformer
has to beat this to justify the extra compute.

In [ ]:
# Baseline — TF-IDF + Logistic Regression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score,
    precision_recall_curve, roc_curve
)

baseline_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=15000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2,
    )),
    ("clf", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        C=1.0,
        solver="lbfgs",
    ))
])

baseline_pipe.fit(train_df["note_text"], train_df["label"])

baseline_probs = baseline_pipe.predict_proba(test_df["note_text"])[:, 1]
baseline_preds = baseline_pipe.predict(test_df["note_text"])

baseline_auc = roc_auc_score(test_df["label"], baseline_probs)
baseline_acc = accuracy_score(test_df["label"], baseline_preds)
baseline_pre = precision_score(test_df["label"], baseline_preds)
baseline_rec = recall_score(test_df["label"], baseline_preds)
baseline_f1  = f1_score(test_df["label"], baseline_preds)

print("=" * 60)
print("  BASELINE: TF-IDF + Logistic Regression (Test Set)")
print("=" * 60)
print(f"  Accuracy : {baseline_acc:.4f}")
print(f"  Precision: {baseline_pre:.4f}")
print(f"  Recall   : {baseline_rec:.4f}")
print(f"  F1 Score : {baseline_f1:.4f}")
print(f"  ROC-AUC  : {baseline_auc:.4f}")
print()
print(classification_report(test_df["label"], baseline_preds,
                             target_names=["Not Ready", "Ready"], digits=3))
cm = confusion_matrix(test_df["label"], baseline_preds)
print("Confusion Matrix:")
print(f"  TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}")

## Tokenization
`MAX_LEN = 128` covers every note (average is about 60 tokens) with room to spare.

In [ ]:
# Tokenization with Final Selected Model (BiomedBERT)
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

# Final selected model (highest performance in our comparison)
# Bio_ClinicalBERT and PubMedBERT are also tokenized inside Cell 7 per-model
model_id = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

# MAX_LEN=128: note_text averages ~60 tokens, so 128 captures everything
# Halves attention computation vs. 256 with zero accuracy loss
MAX_LEN = 128

def tok(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LEN)

def make_ds(df_split):
    return Dataset.from_pandas(
        df_split[['note_text','label']].rename(columns={'note_text':'text'}),
        preserve_index=False
    ).map(tok, batched=True, remove_columns=['text'])

train_ds = make_ds(train_df); train_ds.set_format(type='torch')
val_ds   = make_ds(val_df);   val_ds.set_format(type='torch')
test_ds  = make_ds(test_df);  test_ds.set_format(type='torch')

collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('Tokenization complete. Model:', model_id)
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')
print(f'MAX_LEN: {MAX_LEN} tokens  |  Avg note ~60 tokens — no information lost')

## Fine-tuning
Same settings for all three models so differences come from pre-training only:
learning rate 2e-5, 3 epochs with early stopping, batch size 32, weight decay
0.01, 10% warmup, fp16 on GPU, and class-weighted cross entropy for the 59/41
imbalance. The decision threshold is chosen on the validation PR curve and then
applied unchanged to the test set.

In [ ]:
# Fine-tune Bio_ClinicalBERT, PubMedBERT and BiomedBERT with identical settings
# on the same splits. Roughly 10 to 18 minutes per model on a T4 GPU.

import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer, EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import (
    precision_recall_fscore_support, accuracy_score, roc_auc_score,
    precision_recall_curve
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print("WARNING: No GPU detected. Training will be very slow.")
    print("Go to Runtime > Change runtime type > T4 GPU > Save")

# ── Three models to compare ────────────────────────────────────────────────────
MODELS = {
    "Bio_ClinicalBERT": "emilyalsentzer/Bio_ClinicalBERT",
    "PubMedBERT":       "NLP4Science/pubmedbert-base",
    "BiomedBERT":       "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
}

# ── Class weights (same for all — same training data) ─────────────────────────
y   = train_df["label"].values
pos = (y == 1).sum()
neg = (y == 0).sum()
w0  = pos / (neg + 1e-9)
w1  = 1.0
print(f"Class weights — class 0 (Not Ready): {w0:.3f}, class 1 (Ready): {w1:.3f}")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    try:    auc = roc_auc_score(labels, probs)
    except: auc = float("nan")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1, "roc_auc": auc}

# ── Results store ──────────────────────────────────────────────────────────────
all_results = {}

for model_name, mid in MODELS.items():
    print(f"\n{'='*60}")
    print(f"  Training: {model_name}")
    print(f"  Model ID: {mid}")
    print(f"{'='*60}")

    tok_m = AutoTokenizer.from_pretrained(mid, use_fast=True)
    col_m = DataCollatorWithPadding(tokenizer=tok_m)

    def tok_fn(batch):
        return tok_m(batch["text"], truncation=True, max_length=MAX_LEN)

    def make_ds_m(df_split):
        return Dataset.from_pandas(
            df_split[["note_text","label"]].rename(columns={"note_text":"text"}),
            preserve_index=False
        ).map(tok_fn, batched=True, remove_columns=["text"])

    tr_ds = make_ds_m(train_df); tr_ds.set_format(type="torch")
    v_ds  = make_ds_m(val_df);   v_ds.set_format(type="torch")
    te_ds = make_ds_m(test_df);  te_ds.set_format(type="torch")

    mdl = AutoModelForSequenceClassification.from_pretrained(mid, num_labels=2).to(device)
    cw  = torch.tensor([w0, w1], dtype=torch.float).to(device)

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels  = inputs.get("labels")
            outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
            logits  = outputs.get("logits")
            loss    = torch.nn.CrossEntropyLoss(weight=cw)(logits.view(-1,2), labels.view(-1))
            return (loss, outputs) if return_outputs else loss

    steps_per_epoch = len(tr_ds) // 32
    total_steps     = steps_per_epoch * 3
    warmup_steps    = int(total_steps * 0.10)
    eval_save_steps = max(50, steps_per_epoch // 2)

    args = TrainingArguments(
        output_dir=f"./pa_{model_name.lower().replace('_','')}",
        eval_strategy="steps",  eval_steps=eval_save_steps,
        save_strategy="steps",  save_steps=eval_save_steps,
        save_total_limit=1,
        learning_rate=2e-5,
        num_train_epochs=3,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        gradient_accumulation_steps=1,
        weight_decay=0.01,
        warmup_steps=warmup_steps,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=2,
        dataloader_pin_memory=False,
        logging_steps=50,
        report_to="none",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
    )

    trainer = WeightedTrainer(
        model=mdl, args=args,
        train_dataset=tr_ds, eval_dataset=v_ds,
        data_collator=col_m, compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )
    trainer.train()

    # Threshold optimization on validation set
    val_out    = trainer.predict(v_ds)
    val_probs  = torch.softmax(torch.tensor(val_out.predictions), dim=1).numpy()[:, 1]
    val_labels = val_out.label_ids
    prec_v, rec_v, thr_v = precision_recall_curve(val_labels, val_probs)
    f1s_v      = (2 * prec_v * rec_v) / (prec_v + rec_v + 1e-9)
    best_idx   = int(np.nanargmax(f1s_v))
    best_thr   = float(thr_v[best_idx - 1]) if best_idx > 0 else 0.5

    # Test set evaluation
    te_out    = trainer.predict(te_ds)
    te_probs  = torch.softmax(torch.tensor(te_out.predictions), dim=1).numpy()[:, 1]
    te_labels = te_out.label_ids
    te_preds  = (te_probs >= best_thr).astype(int)
    te_auc    = roc_auc_score(te_labels, te_probs)
    te_acc    = accuracy_score(te_labels, te_preds)
    te_pre, te_rec, te_f1, _ = precision_recall_fscore_support(
        te_labels, te_preds, average="binary", zero_division=0
    )

    all_results[model_name] = {
        "probs": te_probs, "preds": te_preds, "labels": te_labels,
        "thresh": best_thr,
        "probs_val": val_probs, "labels_val": val_labels,
        "prec_val": prec_v, "rec_val": rec_v,
        "acc": te_acc, "pre": te_pre, "rec": te_rec, "f1": te_f1, "auc": te_auc,
    }

    print(f"\n── {model_name} Test Results ──")
    print(f"  Accuracy : {te_acc:.4f}")
    print(f"  Precision: {te_pre:.4f}")
    print(f"  Recall   : {te_rec:.4f}")
    print(f"  F1 Score : {te_f1:.4f}")
    print(f"  ROC-AUC  : {te_auc:.4f}")
    print(f"  Threshold: {best_thr:.4f}")

    # Save BiomedBERT for inference
    if model_name == "BiomedBERT":
        trainer.save_model("./pa_ready_final")
        tok_m.save_pretrained("./pa_ready_final")
        print("  BiomedBERT saved to ./pa_ready_final")
        best_thresh = best_thr
        test_probs  = te_probs
        test_hat    = te_preds
        test_labels = te_labels
        bert_auc    = te_auc

print("\nAll three BERT models trained and evaluated.")

## Saved model
The BiomedBERT checkpoint and tokenizer are written to `./pa_ready_final/` so
inference can run in a fresh session without retraining.

In [ ]:
# Model Saving Confirmation
import os

SAVE_DIR = './pa_ready_final'
if os.path.exists(SAVE_DIR):
    saved_files = os.listdir(SAVE_DIR)
    print(f'BiomedBERT saved to: {SAVE_DIR}')
    print(f'Files saved ({len(saved_files)} total):')
    for f in sorted(saved_files):
        size_kb = os.path.getsize(os.path.join(SAVE_DIR, f)) // 1024
        print(f'  {f:<45} {size_kb:>6} KB')
else:
    print('Save directory not found. Re-run Cell 7.')

print()
print('To reload in a fresh session:')
print("  from transformers import AutoModelForSequenceClassification, AutoTokenizer")
print(f"  model     = AutoModelForSequenceClassification.from_pretrained('{SAVE_DIR}')")
print(f"  tokenizer = AutoTokenizer.from_pretrained('{SAVE_DIR}')")

## ROC and precision-recall curves
All four models on shared axes. The dashed diagonal on the ROC plot is a random
classifier, which is the reference point that matters most for these results.

In [ ]:
# Multi-Model ROC Curves + PR Curves
# Color scheme: grey=baseline, teal=Bio_ClinicalBERT, purple=PubMedBERT, green=BiomedBERT
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix, classification_report, roc_auc_score

COLORS = {
    "TF-IDF Baseline":  "#6B7280",
    "Bio_ClinicalBERT": "#0891B2",
    "PubMedBERT":       "#7C3AED",
    "BiomedBERT":       "#059669",
    "Random":           "#C0C0C0",
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.patch.set_facecolor("#F4F7FA")
for ax in axes:
    ax.set_facecolor("white")
    ax.spines[["top","right"]].set_visible(False)
    ax.grid(alpha=0.25, linewidth=0.5)

# ── ROC CURVE (left) ──────────────────────────────────────────────────────────
axes[0].plot([0,1],[0,1], color=COLORS["Random"], lw=1.5, linestyle="--",
             label="Random  (AUC = 0.500)", alpha=0.7)

fpr_b, tpr_b, _ = roc_curve(test_df["label"].values, baseline_probs)
auc_b = auc(fpr_b, tpr_b)
axes[0].plot(fpr_b, tpr_b, color=COLORS["TF-IDF Baseline"], lw=2.0,
             label=f"TF-IDF Baseline  (AUC = {auc_b:.3f})")

for mname, res in all_results.items():
    fpr, tpr, _ = roc_curve(res["labels"], res["probs"])
    a = auc(fpr, tpr)
    lw = 2.8 if mname == "BiomedBERT" else 2.0
    axes[0].plot(fpr, tpr, color=COLORS[mname], lw=lw,
                 label=f"{mname}  (AUC = {a:.3f})")

axes[0].set_xlabel("False Positive Rate  (1 - Specificity)", fontsize=11)
axes[0].set_ylabel("True Positive Rate  (Sensitivity)", fontsize=11)
axes[0].set_title("ROC Curves — All Models vs. Baseline", fontsize=13, fontweight="bold", pad=10)
axes[0].legend(fontsize=9, loc="lower right")
axes[0].set_xlim(-0.01, 1.01); axes[0].set_ylim(-0.01, 1.01)

# ── PR CURVE (right) ──────────────────────────────────────────────────────────
prec_b, rec_b, _ = precision_recall_curve(test_df["label"].values, baseline_probs)
auc_pr_b = auc(rec_b, prec_b)
axes[1].plot(rec_b, prec_b, color=COLORS["TF-IDF Baseline"], lw=2.0,
             label=f"TF-IDF Baseline  (AUC-PR = {auc_pr_b:.3f})")

for mname, res in all_results.items():
    prec_t, rec_t, _ = precision_recall_curve(res["labels"], res["probs"])
    a_pr = auc(rec_t, prec_t)
    lw = 2.8 if mname == "BiomedBERT" else 2.0
    axes[1].plot(rec_t, prec_t, color=COLORS[mname], lw=lw,
                 label=f"{mname}  (AUC-PR = {a_pr:.3f})")

# Mark BiomedBERT threshold
bert_res = all_results["BiomedBERT"]
prec_v, rec_v, thr_v = precision_recall_curve(bert_res["labels_val"], bert_res["probs_val"])
f1s_v = (2 * prec_v * rec_v) / (prec_v + rec_v + 1e-9)
best_i = int(np.nanargmax(f1s_v))
axes[1].axvline(rec_v[best_i], color=COLORS["BiomedBERT"], linestyle="--", alpha=0.7,
                label=f"BiomedBERT threshold = {bert_res['thresh']:.3f}")

axes[1].set_xlabel("Recall", fontsize=11)
axes[1].set_ylabel("Precision", fontsize=11)
axes[1].set_title("Precision-Recall Curves — All Models", fontsize=13, fontweight="bold", pad=10)
axes[1].legend(fontsize=9, loc="lower left")
axes[1].set_xlim(-0.01, 1.01); axes[1].set_ylim(0.50, 1.02)

plt.suptitle("Prior Authorization Classifier — Model Comparison Curves",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("model_comparison_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# ── BiomedBERT final metrics ───────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  BiomedBERT: test set results")
print("=" * 60)
print(f"  Accuracy : {all_results['BiomedBERT']['acc']:.4f}")
print(f"  Precision: {all_results['BiomedBERT']['pre']:.4f}")
print(f"  Recall   : {all_results['BiomedBERT']['rec']:.4f}")
print(f"  F1 Score : {all_results['BiomedBERT']['f1']:.4f}")
print(f"  ROC-AUC  : {all_results['BiomedBERT']['auc']:.4f}")
print(f"  Threshold: {all_results['BiomedBERT']['thresh']:.4f}")
print()
cm = confusion_matrix(all_results["BiomedBERT"]["labels"], all_results["BiomedBERT"]["preds"])
print("Confusion Matrix:")
print(f"  TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}")
print()
print(classification_report(all_results["BiomedBERT"]["labels"],
                             all_results["BiomedBERT"]["preds"],
                             target_names=["Not Ready","Ready"], digits=3))

## Model comparison
Metric table and grouped bar chart from the computed test results.

In [ ]:
# Full Model Comparison Table + Grouped Bar Chart
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ── Build comparison table ─────────────────────────────────────────────────────
summary_data = {
    "Model": [
        "TF-IDF + LogReg (Baseline)",
        "Bio_ClinicalBERT",
        "PubMedBERT",
        "BiomedNLP-BiomedBERT (selected)",
    ],
    "Accuracy":  [f"{baseline_acc:.4f}", f"{all_results['Bio_ClinicalBERT']['acc']:.4f}",
                  f"{all_results['PubMedBERT']['acc']:.4f}", f"{all_results['BiomedBERT']['acc']:.4f}"],
    "Precision": [f"{baseline_pre:.4f}", f"{all_results['Bio_ClinicalBERT']['pre']:.4f}",
                  f"{all_results['PubMedBERT']['pre']:.4f}", f"{all_results['BiomedBERT']['pre']:.4f}"],
    "Recall":    [f"{baseline_rec:.4f}", f"{all_results['Bio_ClinicalBERT']['rec']:.4f}",
                  f"{all_results['PubMedBERT']['rec']:.4f}", f"{all_results['BiomedBERT']['rec']:.4f}"],
    "F1 Score":  [f"{baseline_f1:.4f}", f"{all_results['Bio_ClinicalBERT']['f1']:.4f}",
                  f"{all_results['PubMedBERT']['f1']:.4f}", f"{all_results['BiomedBERT']['f1']:.4f}"],
    "ROC-AUC":   [f"{baseline_auc:.4f}", f"{all_results['Bio_ClinicalBERT']['auc']:.4f}",
                  f"{all_results['PubMedBERT']['auc']:.4f}", f"{all_results['BiomedBERT']['auc']:.4f}"],
}
summary_df = pd.DataFrame(summary_data)

print("=" * 85)
print("  FULL MODEL COMPARISON — ALL FOUR MODELS (Test Set)")
print("=" * 85)
print(summary_df.to_string(index=False))
print()

b_f1  = all_results["BiomedBERT"]["f1"]
b_auc = all_results["BiomedBERT"]["auc"]
print(f"  BiomedBERT vs Baseline         : +{(b_f1 - baseline_f1)*100:.2f} pp F1,  +{(b_auc - baseline_auc)*100:.2f} pp AUC")
print(f"  BiomedBERT vs Bio_ClinicalBERT : +{(b_f1 - all_results['Bio_ClinicalBERT']['f1'])*100:.2f} pp F1,  +{(b_auc - all_results['Bio_ClinicalBERT']['auc'])*100:.2f} pp AUC")
print(f"  BiomedBERT vs PubMedBERT       : +{(b_f1 - all_results['PubMedBERT']['f1'])*100:.2f} pp F1,  +{(b_auc - all_results['PubMedBERT']['auc'])*100:.2f} pp AUC")

# ── Grouped bar chart of computed test metrics ────────────────────────────────
metric_labels = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"]
model_labels  = ["TF-IDF Baseline", "Bio_ClinicalBERT", "PubMedBERT", "BiomedBERT"]
colors        = ["#94A3B8", "#0891B2", "#7C3AED", "#059669"]

def pct(vals):
    return [round(float(v) * 100, 1) for v in vals]

values = {
    "Accuracy":  pct([baseline_acc] + [all_results[m]["acc"] for m in model_labels[1:]]),
    "Precision": pct([baseline_pre] + [all_results[m]["pre"] for m in model_labels[1:]]),
    "Recall":    pct([baseline_rec] + [all_results[m]["rec"] for m in model_labels[1:]]),
    "F1 Score":  pct([baseline_f1]  + [all_results[m]["f1"]  for m in model_labels[1:]]),
    "ROC-AUC":   pct([baseline_auc] + [all_results[m]["auc"] for m in model_labels[1:]]),
}

x     = np.arange(len(metric_labels))
width = 0.18
fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor("#F4F7FA")
ax.set_facecolor("white")
ax.spines[["top","right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3, linewidth=0.5)

for i, (mname, color) in enumerate(zip(model_labels, colors)):
    vals = [values[m][i] for m in metric_labels]
    bars = ax.bar(x + i * width, vals, width, label=mname,
                  color=color, alpha=0.92, edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.15,
                f"{v:.1f}", ha="center", va="bottom", fontsize=7.5, fontweight="bold")

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0, 105)
ax.set_ylabel("Score (%)", fontsize=11)
ax.set_title("All Models x All Metrics (Test Set)",
             fontsize=12, fontweight="bold", pad=12)
ax.legend(fontsize=10, loc="lower right")

plt.tight_layout()
plt.savefig("all_models_bar_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: all_models_bar_chart.png")

## Error analysis
False positives (predicted Ready, actually Not Ready) and false negatives, the
distribution of P(Ready) for misclassified cases, the cases closest to the
threshold, and breakdowns by lab range status and priority.

In [ ]:
# Qualitative Error Analysis (BiomedBERT)
import matplotlib.pyplot as plt

error_df = test_df.copy().reset_index(drop=True)
error_df["prob_ready"] = all_results["BiomedBERT"]["probs"]
error_df["pred_label"] = all_results["BiomedBERT"]["preds"]
error_df["true_label"] = all_results["BiomedBERT"]["labels"]
error_df["error_type"] = ""

is_fp = (error_df["pred_label"] == 1) & (error_df["true_label"] == 0)
is_fn = (error_df["pred_label"] == 0) & (error_df["true_label"] == 1)
error_df.loc[is_fp, "error_type"] = "False Positive"
error_df.loc[is_fn, "error_type"] = "False Negative"
errors = error_df[error_df["error_type"] != ""].copy()

print(f"Total errors : {len(errors)}")
print(f"  False Positives (FP): {is_fp.sum()}")
print(f"  False Negatives (FN): {is_fn.sum()}")

# ── Two-panel error chart ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor("#F4F7FA")
for ax in axes:
    ax.set_facecolor("white")
    ax.spines[["top","right"]].set_visible(False)

error_counts = errors["error_type"].value_counts()
bar_colors   = {"False Positive": "#B93838", "False Negative": "#B07000"}
bars = axes[0].bar(error_counts.index, error_counts.values,
                   color=[bar_colors.get(x, "#888") for x in error_counts.index],
                   width=0.45, edgecolor="white")
axes[0].set_title("Error Type Distribution — BiomedBERT (Test Set)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Count", fontsize=11)
axes[0].grid(axis="y", alpha=0.3)
for bar, v in zip(bars, error_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.5, str(v),
                 ha="center", fontweight="bold", fontsize=13)
axes[0].set_ylim(0, max(error_counts.values) * 1.25)

thresh = all_results["BiomedBERT"]["thresh"]
for etype, color in [("False Positive","#B93838"), ("False Negative","#B07000")]:
    subset = errors[errors["error_type"] == etype]["prob_ready"]
    axes[1].hist(subset, bins=15, alpha=0.65, color=color, label=etype, edgecolor="white")
axes[1].axvline(thresh, color="#0D1B3E", linestyle="--", lw=2,
                label=f"Decision threshold = {thresh:.3f}")
axes[1].set_title("P(Ready) Distribution of Errors", fontsize=12, fontweight="bold")
axes[1].set_xlabel("P(Ready)", fontsize=11)
axes[1].set_ylabel("Count", fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("error_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Top 15 hardest misclassifications ─────────────────────────────────────────
errors["dist_to_thresh"] = (errors["prob_ready"] - thresh).abs()
hard_cases = errors.sort_values("dist_to_thresh").head(15)

print("\nTop 15 Hardest Misclassifications (closest to decision boundary):")
print("-" * 80)
for _, row in hard_cases.iterrows():
    true_str = "Ready"     if row["true_label"] == 1 else "Not Ready"
    pred_str = "Ready"     if row["pred_label"] == 1 else "Not Ready"
    print(f"[{row['error_type']:16s}]  True: {true_str:9s}  Pred: {pred_str:9s}  "
          f"P(Ready)={row['prob_ready']:.3f}")
    print(f"  Note: {row['note_text'][:110]}...")
    print()

print("\nError breakdown by lab range_status:")
print(errors.groupby(["error_type","range_status"]).size().unstack(fill_value=0))
print("\nError breakdown by priority:")
print(errors.groupby(["error_type","priority_clean"]).size().unstack(fill_value=0))

## Inference
Reloads the saved model from disk and scores new notes. `confidence` is
`max(p, 1 - p)`; anything near 50% should go to a human reviewer.

In [ ]:
# Inference Function — Load Saved BiomedBERT + Predict
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

SAVE_DIR = "./pa_ready_final"
infer_model     = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(device)
infer_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
infer_model.eval()

# Use the validation-optimized threshold from Cell 7
best_thresh = all_results["BiomedBERT"]["thresh"]

def predict_pa_ready(note_text, threshold=None):
    if threshold is None:
        threshold = best_thresh
    inputs = infer_tokenizer(
        note_text, return_tensors="pt",
        truncation=True, max_length=MAX_LEN
    ).to(device)
    with torch.no_grad():
        logits     = infer_model(**inputs).logits
        prob_ready = torch.softmax(logits, dim=1)[0, 1].item()
    pred = int(prob_ready >= threshold)
    return {
        "prob_ready":  round(prob_ready, 4),
        "pred_label":  pred,
        "pred_str":    "Ready" if pred == 1 else "Not Ready",
        "confidence":  f"{max(prob_ready, 1-prob_ready)*100:.1f}%"
    }

# ── Example predictions ────────────────────────────────────────────────────────
examples = [
    (
        "Diagnosis Hypertension. Procedure Echocardiogram. "
        "Lab item 50868 value 1.2 mg/dL. Reference range 0.6 to 1.3. "
        "Severity within normal range. Flag normal. Priority routine.",
        "Expected: Ready (normal labs, routine priority)"
    ),
    (
        "Diagnosis Asthma uncontrolled. Procedure Pulmonary function test. "
        "Lab item 50931 value 250.0 mg/dL. Reference range 70.0 to 100.0. "
        "Severity critically above range. Flag abnormal. Priority stat.",
        "Expected: Not Ready (critically out of range)"
    ),
    (
        "Diagnosis Type 2 Diabetes. Procedure Continuous glucose monitor. "
        "Lab item 50809 value 108.0 mg/dL. Reference range 70.0 to 100.0. "
        "Severity mildly above range. Flag abnormal. Priority routine.",
        "Expected: boundary case (mildly abnormal — model uncertainty zone)"
    ),
]

print(f"BiomedBERT Inference  |  Decision threshold: {best_thresh:.4f}")
print("=" * 65)
for ex_text, expected in examples:
    result = predict_pa_ready(ex_text)
    print(f"\nInput   : {ex_text[:95]}...")
    print(f"Expected: {expected}")
    print(f"Result  : {result['pred_str']}  |  P(Ready)={result['prob_ready']}  "
          f"|  Confidence={result['confidence']}")

## Summary
See the README for the test results and discussion. In short: after removing
the template leakage, none of the models (including the TF-IDF baseline)
separated Ready from Not Ready better than chance on ROC-AUC. The transformers
reached a higher F1 mainly by predicting Ready for almost every case. The next
step is to check whether the remaining fields carry any signal about the label
at all before spending more effort on model choice.